In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, DataLoader

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn import preprocessing

feature_set = ['Active Power', 'Reactive Power', 'Governor speed actual', 'UGB X displacement', 'UGB Y displacement',
    'LGB X displacement', 'LGB Y displacement', 'TGB X displacement',
    'TGB Y displacement', 'Stator winding temperature 13',
    'Stator winding temperature 14', 'Stator winding temperature 15',
    'Surface Air Cooler Air Outlet Temperature',
    'Surface Air Cooler Water Inlet Temperature',
    'Surface Air Cooler Water Outlet Temperature',
    'Stator core temperature', 'UGB metal temperature',
    'LGB metal temperature 1', 'LGB metal temperature 2',
    'LGB oil temperature', 'Penstock Flow', 'Turbine flow',
    'UGB cooling water flow', 'LGB cooling water flow',
    'Generator cooling water flow', 'Governor Penstock Pressure',
    'Penstock pressure', 'Opening Wicked Gate', 'UGB Oil Contaminant',
    'Gen Thrust Bearing Oil Contaminant']

def fill_nans_conditionally(series, threshold=100):
    is_nan = series.isna()
    groups = (is_nan != is_nan.shift()).cumsum()
    
    nan_counts = is_nan.groupby(groups).transform('sum')
    series_filled = series.copy()
    series_filled[nan_counts < threshold] = series_filled[nan_counts < threshold].fillna(method='ffill')
    return series_filled

def consecutive_nan_info(df, col, timestamp_col='TimeStamp'):
    is_nan = df[col].isna()
    groups = (is_nan != is_nan.shift()).cumsum()
    
    results = []
    for group_id, group_data in df.groupby(groups):
        if group_data[col].isna().all():
            count = group_data.shape[0]
            start_time = group_data[timestamp_col].iloc[0]
            end_time = group_data[timestamp_col].iloc[-1]
            results.append({"start_time": start_time, "end_time": end_time, "nan_count": count})
    return results


df_data_withtime = pd.read_csv("Data20212025.csv")
df_data_withtime['TimeStamp'] = pd.to_datetime(df_data_withtime['TimeStamp'])
df_data_withtime = df_data_withtime[['TimeStamp'] + feature_set]
df_data_withtime = df_data_withtime.sort_values(by='TimeStamp', ascending=True)  

is_ordered = df_data_withtime.index.is_monotonic_increasing
if not is_ordered:
    disorder = df_data_withtime.index.to_series().diff().fillna(0) < 0
    unordered_positions = disorder[disorder].index.tolist()
    print("Index is not ordered. Disorder at positions (index values):", unordered_positions)
else:
    print("Index is in order.")

df_data_withtime = df_data_withtime.reset_index(drop=True)

Index is in order.


In [2]:
df_anomaly = pd.read_excel("/run/media/fourier/Data2/Pras/Vale/time-series-autoencoder/shutdown_list.xlsx", 'Sheet2')
df_anomaly['Start Time'] = pd.to_datetime(df_anomaly['Start Time'])
df_anomaly['End Time'] = pd.to_datetime(df_anomaly['End Time'])

df_anomaly_unplaned = df_anomaly.copy()
mask = (df_anomaly_unplaned['Interal/External'] == 'Internal') & (df_anomaly_unplaned['Shutdown Type'] == 'Unplanned') & (df_anomaly_unplaned['Start Time'] >= '2020-01-01 00:00:00')
df_anomaly_unplaned = df_anomaly_unplaned.loc[mask]
df_anomaly_unplaned = df_anomaly_unplaned.drop(df_anomaly_unplaned.index[[2]])
df_anomaly_unplaned = df_anomaly_unplaned.reset_index(drop=True)

In [3]:
for index, row in df_anomaly_unplaned.iterrows():
    masknot = (df_data_withtime['TimeStamp'] > (row['Start Time'] - timedelta(hours=24 * 7))) & (df_data_withtime['TimeStamp'] <= (row['End Time'] + timedelta(hours=24 * 7)))
    df_data_withtime = df_data_withtime.loc[~masknot]

In [4]:
def process_shutdownTimestamp(data_timestamp, sensor_datas):
    activepower_data = sensor_datas[:, 0].astype(float)
    rpm_data = sensor_datas[:, 2].astype(float)
    shutdown_mask = (activepower_data <= 3) & (rpm_data <= 10)
    change_points = np.diff(shutdown_mask.astype(int), prepend=0)

    start_indices = np.where(change_points == 1)[0]
    end_indices = np.where(change_points == -1)[0]

    if shutdown_mask[-1]:
        end_indices = np.append(end_indices, len(shutdown_mask))

    if shutdown_mask[0]:
        start_indices = np.insert(start_indices, 0, 0)

    shutdown_periods = []
    for start, end in zip(start_indices, end_indices):
        start_time = data_timestamp[start]
        end_time = data_timestamp[end - 1]
        shutdown_periods.append((start_time, end_time))
    
    return shutdown_periods

In [5]:
print(df_data_withtime.shape)

(2784175, 31)


In [6]:
data_timestamp = df_data_withtime[['TimeStamp']].values
shutdown_periods = process_shutdownTimestamp(data_timestamp, df_data_withtime.values[:, 1:5])

for start_time, end_time in shutdown_periods:
    start_time = pd.Timestamp(start_time[0])
    end_time = pd.Timestamp(end_time[0])

    masknot = (df_data_withtime['TimeStamp'] > (start_time - timedelta(hours=24))) & (df_data_withtime['TimeStamp'] <= (end_time + timedelta(hours=24)))
    df_data_withtime = df_data_withtime.loc[~masknot]

print(df_data_withtime.shape)

(2465874, 31)


In [7]:
sensor_columns = [col for col in df_data_withtime.columns if col not in ["TimeStamp"]]
nan_info_all = {}

for col in sensor_columns:
    nan_info_all[col] = consecutive_nan_info(df_data_withtime, col)

for sensor, info in nan_info_all.items():
    for group in info:
        print(group)

In [8]:
df_data_withtime = df_data_withtime.dropna()
print(df_data_withtime.shape)

(2465874, 31)


In [9]:
df_data_withtime = df_data_withtime.sort_values(by='TimeStamp')
mask = (df_data_withtime['TimeStamp'] <= '2025-03-28 00:00:00')
df_data_withtime = df_data_withtime.loc[mask]
df_data_withtime.reset_index(drop=True, inplace=True)

In [10]:
def filter_noise_es(df, alpha=0.4, reduction=False):
    import copy
    new_df = copy.deepcopy(df)
    
    for column in df:
        new_df[column] = df[column].ewm(alpha=alpha, adjust=False).mean()
    
    if reduction:
        return new_df[::len(df)]  # Adjust sparsity if needed
    else:
        return new_df

def wgn_pandas(df_withtime, snr, alpha=0.15, window_size=120):
    df_no_timestamp = df_withtime.drop(columns=['TimeStamp'])
    noisy_df = pd.DataFrame(index=df_no_timestamp.index, columns=df_no_timestamp.columns)

    for start in range(0, len(df_no_timestamp), window_size):
        window = df_no_timestamp.iloc[start:start + window_size]
        
        min_window, max_window = window.min(), window.max()
        #x = (window - min_window) / (max_window - min_window + 1e-4)
        Ps = np.sum(np.power(window, 2), axis=0) / len(window)
        Pn = Ps / (np.power(10, snr / 10))

        noise = np.random.randn(*window.shape) * np.sqrt(Pn.values)
        noisy_window = window + (noise / 100)

        noisy_df.iloc[start:start + window_size] = noisy_window
    
    noisy_df.reset_index(drop=True, inplace=True)
    noisy_df = filter_noise_es(pd.DataFrame(noisy_df, columns=noisy_df.columns), alpha)

    df_timestamp = df_withtime['TimeStamp']
    df_timestamp.reset_index(drop=True, inplace=True)

    df_withtime = pd.concat([df_timestamp, noisy_df], axis=1)
    return df_withtime

df_noisy_wgn = wgn_pandas(df_data_withtime, 30, alpha=0.15)

In [11]:
df_label = pd.DataFrame({
    'TimeStamp': df_data_withtime['TimeStamp'],
    'label_anomaly': 0
})

In [12]:
new_df = df_noisy_wgn
sampels = int(len(new_df) * 0.85)
train_data = new_df[:sampels]
test_data = new_df[sampels:]
test_label = df_label[sampels:]

train_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)
test_label.reset_index(drop=True, inplace=True)

train_data.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/Custom202020253AWGN30ES15/train.csv", index=False)
test_data.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/Custom202020253AWGN30ES15/test.csv", index=False)
test_label.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/Custom202020253AWGN30ES15/test_label.csv", index=False)

# Combine Datasets

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, DataLoader

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn import preprocessing

feature_set = ['Active Power', 'Reactive Power', 'Governor speed actual', 'UGB X displacement', 'UGB Y displacement',
    'LGB X displacement', 'LGB Y displacement', 'TGB X displacement',
    'TGB Y displacement', 'Stator winding temperature 13',
    'Stator winding temperature 14', 'Stator winding temperature 15',
    'Surface Air Cooler Air Outlet Temperature',
    'Surface Air Cooler Water Inlet Temperature',
    'Surface Air Cooler Water Outlet Temperature',
    'Stator core temperature', 'UGB metal temperature',
    'LGB metal temperature 1', 'LGB metal temperature 2',
    'LGB oil temperature', 'Penstock Flow', 'Turbine flow',
    'UGB cooling water flow', 'LGB cooling water flow',
    'Generator cooling water flow', 'Governor Penstock Pressure',
    'Penstock pressure', 'Opening Wicked Gate', 'UGB Oil Contaminant',
    'Gen Thrust Bearing Oil Contaminant']

In [ ]:
# (df_2023.index == sorted(df_2023.index)).all()

In [17]:
df_2023 = pd.read_pickle("/run/media/fourier/Data2/Pras/Vale/time-series-autoencoder/my_data_5thn_olah.pickle")
mask = (df_2023['TimeStamp'] >= '2020-01-01 00:00:00')
df_2023 = df_2023.loc[mask]
df_2023['TimeStamp'] = pd.to_datetime(df_2023['TimeStamp'])

for column_name in df_2023.columns:
    if column_name != 'Load_Type' and column_name != 'TimeStamp':
        df_2023[column_name] = pd.to_numeric(df_2023[column_name], downcast='float')

df_2023 = df_2023.sort_values(by='TimeStamp', ascending=True)  
df_2023 = df_2023.reset_index(drop=True) 

df_2024 = pd.read_csv("PI2024-Now.csv")
for column_name in df_2024.columns:
    if column_name != 'Load_Type' and column_name != 'TimeStamp':
        df_2024[column_name] = pd.to_numeric(df_2024[column_name], downcast='float')
df_2024['TimeStamp'] = pd.to_datetime(df_2024['TimeStamp'])
mask = (df_2024['TimeStamp'] >= '2024-01-01 00:00:00')
df_2024 = df_2024.loc[mask]
df_2024 = df_2024.sort_values(by='TimeStamp', ascending=True)  
df_2024 = df_2024.reset_index(drop=True) 

df_2025 = pd.read_csv("PI2025-Now.csv")
for column_name in df_2025.columns:
    if column_name != 'Load_Type' and column_name != 'TimeStamp':
        df_2025[column_name] = pd.to_numeric(df_2025[column_name], downcast='float')
df_2025['TimeStamp'] = pd.to_datetime(df_2025['TimeStamp'])
df_2024 = df_2024.sort_values(by='TimeStamp', ascending=True)  
df_2024 = df_2024.reset_index(drop=True) 

df_2023 = df_2023[['TimeStamp'] + feature_set]
df_2024 = df_2024[['TimeStamp'] + feature_set]
df_2025 = df_2025[['TimeStamp'] + feature_set]

In [21]:
result_rows = pd.concat([df_2023, df_2024, df_2025])
result_rows = result_rows.sort_values(by='TimeStamp', ascending=True)  
result_rows = result_rows.reset_index(drop=True) 

In [22]:
result_rows.to_csv("Data20212025.csv", index=False)

In [23]:
result_rows

,TimeStamp,Active Power,Reactive Power,Governor speed actual,UGB X displacement,UGB Y displacement,LGB X displacement,LGB Y displacement,TGB X displacement,TGB Y displacement,...,Penstock Flow,Turbine flow,UGB cooling water flow,LGB cooling water flow,Generator cooling water flow,Governor Penstock Pressure,Penstock pressure,Opening Wicked Gate,UGB Oil Contaminant,Gen Thrust Bearing Oil Contaminant
0,2020-01-01 00:00:00,42.880001,18.790001,276.959991,174.589996,172.720001,128.589996,108.400002,167.479996,129.119995,...,36.759998,32.439999,29.570000,1196.760010,3385.090088,276.959991,13.850000,64.910004,15.420000,20.200001
1,2020-01-01 00:01:00,43.650002,22.850000,274.690002,183.960007,170.679993,129.889999,110.900002,181.380005,134.479996,...,36.630001,32.130001,29.570000,1195.819946,3381.300049,274.850006,13.320000,65.400002,15.420000,20.200001
2,2020-01-01 00:02:00,44.419998,20.129999,275.079987,150.539993,162.710007,132.789993,104.589996,165.520004,119.209999,...,36.529999,31.809999,29.570000,1197.540039,3383.780029,275.359985,13.340000,65.879997,15.420000,20.200001
3,2020-01-01 00:03:00,45.189999,17.410000,275.869995,173.110001,177.080002,121.610001,112.690002,183.300003,136.990005,...,36.419998,31.500000,29.570000,1196.319946,3384.389893,275.869995,13.810000,64.599998,15.420000,20.200001
4,2020-01-01 00:04:00,45.959999,22.230000,274.260010,180.100006,164.110001,130.389999,98.800003,170.889999,132.100006,...,36.320000,31.190001,29.570000,1198.319946,3382.800049,274.260010,13.830000,64.800003,15.420000,20.200001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2847455,2025-06-02 09:34:00,41.845524,23.571930,274.117279,174.623581,194.893066,136.477081,92.786644,91.537727,63.547947,...,33.849476,27.350132,28.193481,866.875061,3522.499756,274.023468,13.950585,61.079411,16.913958,19.292192
2847456,2025-06-02 09:35:00,41.847446,23.578930,274.116943,190.386887,180.408264,117.000237,98.493240,68.503143,66.800163,...,33.849236,27.403399,28.195316,865.448364,3541.708008,274.023102,13.951126,61.068413,16.915419,19.294527
2847457,2025-06-02 09:36:00,41.849365,23.585932,274.116577,181.925415,181.542740,124.493774,91.498657,84.382835,71.892258,...,33.848999,27.456665,28.197151,868.439087,3443.270996,274.022736,13.951667,61.057415,16.916880,19.296860
2847458,2025-06-02 09:37:00,41.851284,23.592932,274.122650,199.996124,191.404160,113.810234,94.498337,75.919914,61.106640,...,33.848763,27.509932,28.198986,864.455383,3474.916748,274.022369,13.952209,61.046421,16.918341,19.299194


In [37]:
is_ordered = df_2023.index.is_monotonic_increasing
if not is_ordered:
    disorder = df_2023.index.to_series().diff().fillna(0) < 0
    unordered_positions = disorder[disorder].index.tolist()
    print("Index is not ordered. Disorder at positions (index values):", unordered_positions)
else:
    print("Index is in order.")

Index is not ordered. Disorder at positions (index values): [1440, 2880, 4320]


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, DataLoader

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn import preprocessing

feature_set = ['Active Power', 'Reactive Power', 'Governor speed actual', 'UGB X displacement', 'UGB Y displacement',
    'LGB X displacement', 'LGB Y displacement', 'TGB X displacement',
    'TGB Y displacement', 'Stator winding temperature 13',
    'Stator winding temperature 14', 'Stator winding temperature 15',
    'Surface Air Cooler Air Outlet Temperature',
    'Surface Air Cooler Water Inlet Temperature',
    'Surface Air Cooler Water Outlet Temperature',
    'Stator core temperature', 'UGB metal temperature',
    'LGB metal temperature 1', 'LGB metal temperature 2',
    'LGB oil temperature', 'Penstock Flow', 'Turbine flow',
    'UGB cooling water flow', 'LGB cooling water flow',
    'Generator cooling water flow', 'Governor Penstock Pressure',
    'Penstock pressure', 'Opening Wicked Gate', 'UGB Oil Contaminant',
    'Gen Thrust Bearing Oil Contaminant']

def fill_nans_conditionally(series, threshold=100):
    is_nan = series.isna()
    groups = (is_nan != is_nan.shift()).cumsum()
    
    nan_counts = is_nan.groupby(groups).transform('sum')
    series_filled = series.copy()
    series_filled[nan_counts < threshold] = series_filled[nan_counts < threshold].fillna(method='ffill')
    return series_filled

def consecutive_nan_info(df, col, timestamp_col='TimeStamp'):
    is_nan = df[col].isna()
    groups = (is_nan != is_nan.shift()).cumsum()
    
    results = []
    for group_id, group_data in df.groupby(groups):
        if group_data[col].isna().all():
            count = group_data.shape[0]
            start_time = group_data[timestamp_col].iloc[0]
            end_time = group_data[timestamp_col].iloc[-1]
            results.append({"start_time": start_time, "end_time": end_time, "nan_count": count})
    return results

In [2]:
df_data_withtime = pd.read_csv("labeled_dataset_2024.csv")
df_data_withtime['TimeStamp'] = pd.to_datetime(df_data_withtime['TimeStamp'])
df_data_withtime = df_data_withtime[['TimeStamp'] + feature_set]
df_data_withtime = df_data_withtime.reset_index(drop=True)

In [3]:
sensor_columns = [col for col in df_data_withtime.columns if col != "TimeStamp"]
df_data_withtime[sensor_columns] = df_data_withtime[sensor_columns].apply(lambda col: fill_nans_conditionally(col))

In [4]:
df_label = pd.DataFrame({
    'TimeStamp': df_data_withtime['TimeStamp'],
    'label_anomaly': 0
})

In [5]:
df_anomaly = pd.read_excel("/run/media/fourier/Data2/Pras/Vale/time-series-autoencoder/shutdown_list.xlsx", 'Sheet2')
df_anomaly['Start Time'] = pd.to_datetime(df_anomaly['Start Time'])
df_anomaly['End Time'] = pd.to_datetime(df_anomaly['End Time'])

for index, row in df_anomaly.iterrows():
    masknot = (df_data_withtime['TimeStamp'] > (row['Start Time'] - timedelta(hours=24 * 7))) & (df_data_withtime['TimeStamp'] <= (row['End Time'] + timedelta(hours=24 * 7)))
    df_data_withtime = df_data_withtime.loc[~masknot]
    
# Create df_anomaly with maintenance events
df_anomaly_tangan = pd.DataFrame({
    'Start Time': [
        datetime(2024, 7, 9, 10, 48),
        datetime(2024, 10, 15, 1, 28),
        datetime(2024, 10, 30, 10, 15)
    ],
    'End Time': [
        datetime(2024, 7, 9, 12, 7),
        datetime(2024, 10, 15, 2, 12),
        datetime(2024, 10, 30, 12, 0)
    ],
    'Event': [
        'replace carbon brush',
        'check reducing valve',
        'clearn up slip ring housing'
    ]
})

# Apply the filtering code
for index, row in df_anomaly_tangan.iterrows():
    masknot = (df_data_withtime['TimeStamp'] > (row['Start Time'] - timedelta(hours=24 * 7))) & \
              (df_data_withtime['TimeStamp'] <= (row['End Time'] + timedelta(hours=24 * 7)))
    df_data_withtime = df_data_withtime.loc[~masknot]

# Print remaining data length
print(len(df_data_withtime))

1820845


In [6]:
start_extended = pd.Timestamp("2022-06-01 00:00:00") - pd.Timedelta(days=7)
end_extended = pd.Timestamp("2022-06-12 00:00:00") + pd.Timedelta(days=7)

mask = ~df_data_withtime['TimeStamp'].between(start_extended, end_extended)
df_data_withtime = df_data_withtime[mask]


In [8]:
mask = (df_data_withtime['TimeStamp'] >= '2020-01-01 00:00:00') & (df_data_withtime['TimeStamp'] < '2024-12-08 12:20:55')
df_data_withtime = df_data_withtime.loc[mask]
print(len(df_data_withtime) / 525600)

2.7175551750380516


In [11]:
sensor_columns = [col for col in df_data_withtime.columns if col not in ["TimeStamp"]]
nan_info_all = {}

for col in sensor_columns:
    nan_info_all[col] = consecutive_nan_info(df_data_withtime, col)

for sensor, info in nan_info_all.items():
    for group in info:
        print(group)

In [12]:
df_data_withtime = df_data_withtime.dropna()

In [15]:
df_data_withtime = df_data_withtime.sort_values(by='TimeStamp')
df_data_withtime.reset_index(drop=True, inplace=True)

In [17]:
def filter_noise_es(df, alpha=0.4, reduction=False):
    import copy
    new_df = copy.deepcopy(df)
    
    for column in df:
        new_df[column] = df[column].ewm(alpha=alpha, adjust=False).mean()
    
    if reduction:
        return new_df[::len(df)]  # Adjust sparsity if needed
    else:
        return new_df

def wgn_pandas(df_withtime, snr, alpha=0.15, window_size=120):
    df_no_timestamp = df_withtime.drop(columns=['TimeStamp'])
    noisy_df = pd.DataFrame(index=df_no_timestamp.index, columns=df_no_timestamp.columns)

    for start in range(0, len(df_no_timestamp), window_size):
        window = df_no_timestamp.iloc[start:start + window_size]
        
        min_window, max_window = window.min(), window.max()
        #x = (window - min_window) / (max_window - min_window + 1e-4)
        Ps = np.sum(np.power(window, 2), axis=0) / len(window)
        Pn = Ps / (np.power(10, snr / 10))

        noise = np.random.randn(*window.shape) * np.sqrt(Pn.values)
        noisy_window = window + (noise / 100)

        noisy_df.iloc[start:start + window_size] = noisy_window
    
    noisy_df.reset_index(drop=True, inplace=True)
    noisy_df = filter_noise_es(pd.DataFrame(noisy_df, columns=noisy_df.columns), alpha)

    df_timestamp = df_withtime['TimeStamp']
    df_timestamp.reset_index(drop=True, inplace=True)

    df_withtime = pd.concat([df_timestamp, noisy_df], axis=1)
    return df_withtime

df_noisy_wgn = wgn_pandas(df_data_withtime, 30, alpha=0.15)

In [18]:
len(df_noisy_wgn)

1428347

In [19]:
for _, row in df_anomaly.iterrows():
    start_time = row['Start Time']
    end_time = row['End Time']
    pre_start_time = start_time - pd.Timedelta(hours=3)
    
    df_label.loc[
        (df_label['TimeStamp'] >= pre_start_time) & 
        (df_label['TimeStamp'] < start_time), 
        'label_anomaly'
    ] = 1
    
df_label.reset_index(drop=True, inplace=True)

In [25]:
new_df = df_noisy_wgn
sampels = int(len(new_df) * 0.9)
train_data = new_df[:sampels]
test_data = new_df[sampels:]
test_label = df_label[sampels:]

train_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)
test_label.reset_index(drop=True, inplace=True)

train_data.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/Custom2024AWGN30ES15/train.csv", index=False)
test_data.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/Custom2024AWGN30ES15/test.csv", index=False)
test_label.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/Custom2024AWGN30ES15/test_label.csv", index=False)

# LGS 2

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, DataLoader

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn import preprocessing

import matplotlib.pyplot as plt

def label_load(row):
   if row['Active Power'] < 1 and row['Governor speed actual'] < 1:
      return 'Shutdown'
   elif row['Active Power'] < 3 and row['Governor speed actual'] < 250:
      return 'Warming'
   elif row['Active Power'] < 3 and row['Governor speed actual'] > 250:
      return 'No Load'
   elif row['Active Power'] >= 1 and row['Active Power'] < 20 and row['Governor speed actual'] > 250:
      return 'Low Load'
   elif row['Active Power'] >= 20 and row['Active Power'] < 40 and row['Governor speed actual'] > 250:
      return 'Rough Zone'
   elif row['Active Power'] >= 40 and row['Active Power'] < 50 and row['Governor speed actual'] > 250:
      return 'Part Load'
   elif row['Active Power'] >= 50 and row['Active Power'] < 65 and row['Governor speed actual'] > 250:
      return 'Efficient Load'
   elif row['Active Power'] >= 65 and row['Governor speed actual'] > 250:
      return 'High Load'
   else:
      return 'Undefined'
      
class TimeSeriesDataset(Dataset):
    def __init__(self, input_data, input_window, output_window):
        self.input_data = input_data
        self.input_window = input_window
        self.output_window = output_window
        self.block_len = input_window + output_window
        self.block_num = len(input_data) - self.block_len + 1
        self.inout_seq = self.create_input_sequences()

    def create_input_sequences(self):
        inout_seq = []
        for i in range(self.block_num):
            train_seq = self.input_data[i : i + self.input_window]
            train_label = self.input_data[i + self.output_window : i + self.input_window + self.output_window][:, 9:16]
            inout_seq.append((train_seq, train_label))
        return inout_seq

    def __len__(self):
        return len(self.inout_seq)

    def __getitem__(self, idx):
        train_seq, train_label = self.inout_seq[idx]
        return torch.FloatTensor(train_seq), torch.FloatTensor(train_label)

In [2]:
df_data_withtime = pd.read_pickle("/run/media/fourier/Data2/Pras/Vale/Data_Raw/LGS2/lgs2_olah.pickle")
mask = (df_data_withtime['TimeStamp'] >= '2020-01-01 00:00:00')
df_data_withtime = df_data_withtime.loc[mask]
df_data_withtime['TimeStamp'] = pd.to_datetime(df_data_withtime['TimeStamp'])

for column_name in df_data_withtime.columns:
    if column_name != 'Load_Type' and column_name != 'TimeStamp':
        df_data_withtime[column_name] = pd.to_numeric(df_data_withtime[column_name], downcast='float')

df_data_withtime = df_data_withtime.fillna(method='ffill')
print(len(df_data_withtime))

2103840


In [3]:
df_data_withtime.columns

Index(['TimeStamp', 'U-Lgs2-Ti-81204D-Ai',
       'Surface Air Cooler Water Inlet Temp',
       'Surface Air  Cooler Water Outlet Temp', 'Governor Unit Speed Actual',
       'Upper Guide Bearing X Vibration', 'Upper Guide Bearing Y Vibration',
       'Lower Guide Bearing X Vibration', 'Lower Guide Bearing Y Vibration',
       'Turbine Guide Bearing X Vibration',
       'Turbine Guide Bearing Y Vibration', 'Gen Voltage Ph 1',
       'Gen Voltage Ph 2', 'Gen Voltage Ph 3', 'Gen Current Ph 1',
       'Gen Current Ph 2', 'Gen Current Ph 3', 'Active Power ',
       'Reactive Power', 'Excitation Field Voltage',
       'Excitation Field Current', 'Gen Frequency', 'Power Factor (Modbus)',
       'Turb Gov Turbine Wicket Gate Position (%)', 'Penstock Pressure',
       'Governor Penstock Pressure', 'Penstock Flow', 'Gov Turbine Flow',
       'Efficiency', 'Generator Coolers Water Flow ',
       'Upper Guide Bearing Cooling Water Flow',
       'Lower Combined Bearing Cooling Water Flow ',
       

In [8]:
df_data_withtime.head()

,TimeStamp,U-Lgs2-Ti-81204D-Ai,Surface Air Cooler Water Inlet Temp,Surface Air Cooler Water Outlet Temp,Governor Unit Speed Actual,Upper Guide Bearing X Vibration,Upper Guide Bearing Y Vibration,Lower Guide Bearing X Vibration,Lower Guide Bearing Y Vibration,Turbine Guide Bearing X Vibration,...,L_U2_Gov_Bypass Valve Position,Stator Winding Rtd #13,Stator Winding Rtd #14,Stator Winding Rtd #15,Stator Core Rtd #12 Temp Air Outlet Temp,Upper Guide Bearing Metal Rtd #3 Air Outlet Temp,Upper Guide Bearing Oil Rtd,Lower Guide Bearing Metal Rtd #1,Lower Guide Bearing Metal Rtd #2,Lower Guide Bearing Oil Rtd
0,2020-01-01 00:00:00,70.0,68.43,70.0,274.359985,306.480011,295.019989,115.620003,104.019997,105.599998,...,-0.18,70.0,70.419998,69.470001,59.369999,52.0,51.0,56.0,55.0,72.0
1,2020-01-01 00:01:00,70.0,68.43,70.0,274.359985,290.940002,282.739990,138.580002,133.490005,114.940002,...,-0.18,70.0,70.419998,69.470001,59.360001,52.0,51.0,56.0,55.0,72.0
2,2020-01-01 00:02:00,70.0,68.43,70.0,274.359985,300.260010,274.929993,121.809998,118.300003,144.520004,...,-0.18,70.0,70.419998,69.480003,59.360001,52.0,51.0,56.0,55.0,72.0
3,2020-01-01 00:03:00,70.0,68.43,70.0,274.359985,309.880005,291.089996,114.800003,120.279999,133.279999,...,-0.18,70.0,70.419998,69.480003,59.360001,52.0,51.0,56.0,55.0,72.0
4,2020-01-01 00:04:00,70.0,68.43,70.0,274.359985,308.910004,295.790009,130.809998,116.410004,103.099998,...,-0.18,70.0,70.419998,69.480003,59.360001,52.0,51.0,56.0,55.0,72.0


In [9]:
df_anomaly = pd.read_excel("/run/media/fourier/Data2/Pras/Vale/Data_Raw/LGS2/shutdown_lgs2.xlsx")
df_anomaly['Start Time'] = pd.to_datetime(df_anomaly['Start Time'])
df_anomaly['End Time'] = pd.to_datetime(df_anomaly['End Time'])

# mask = (df_anomaly['Interal/External'] == 'Internal') & (df_anomaly['Shutdown Type'] == 'Unplanned')
# df_anomaly = df_anomaly.loc[~mask]

for index, row in df_anomaly.iterrows():
    masknot = (df_data_withtime['TimeStamp'] > (row['Start Time'] - timedelta(hours=24 * 7))) & (df_data_withtime['TimeStamp'] <= (row['End Time'] + timedelta(hours=24 * 7)))
    df_data_withtime = df_data_withtime.loc[~masknot]
    
print(len(df_data_withtime))

1895590


In [11]:
(2103840 - 1895590) / 60 / 24

144.61805555555557

In [13]:
df_data_withtime.reset_index(drop=True, inplace=True)

In [14]:
def filter_noise_es(df, alpha=0.4, reduction=False):
    import copy
    new_df = copy.deepcopy(df)
    
    for column in df:
        new_df[column] = df[column].ewm(alpha=alpha, adjust=False).mean()
    
    if reduction:
        return new_df[::len(df)]  # Adjust sparsity if needed
    else:
        return new_df

def wgn_pandas(df_withtime, snr, alpha=0.15, window_size=120):
    df_no_timestamp = df_withtime.drop(columns=['TimeStamp'])
    noisy_df = pd.DataFrame(index=df_no_timestamp.index, columns=df_no_timestamp.columns)

    for start in range(0, len(df_no_timestamp), window_size):
        window = df_no_timestamp.iloc[start:start + window_size]
        
        min_window, max_window = window.min(), window.max()
        #x = (window - min_window) / (max_window - min_window + 1e-4)
        Ps = np.sum(np.power(window, 2), axis=0) / len(window)
        Pn = Ps / (np.power(10, snr / 10))

        noise = np.random.randn(*window.shape) * np.sqrt(Pn.values)
        noisy_window = window + (noise / 100)

        noisy_df.iloc[start:start + window_size] = noisy_window
    
    noisy_df.reset_index(drop=True, inplace=True)
    noisy_df = filter_noise_es(pd.DataFrame(noisy_df, columns=noisy_df.columns), alpha)

    df_timestamp = df_withtime['TimeStamp']
    df_timestamp.reset_index(drop=True, inplace=True)

    df_withtime = pd.concat([df_timestamp, noisy_df], axis=1)
    return df_withtime

df_noisy_wgn = wgn_pandas(df_data_withtime, 30, alpha=0.15)

In [18]:
feature_set = ['Active Power', 'Reactive Power', 'Governor speed actual', 'UGB X displacement', 'UGB Y displacement',
    'LGB X displacement', 'LGB Y displacement', 'TGB X displacement',
    'TGB Y displacement', 'Stator winding temperature 13',
    'Stator winding temperature 14', 'Stator winding temperature 15',
    'Surface Air Cooler Air Outlet Temperature',
    'Surface Air Cooler Water Inlet Temperature',
    'Surface Air Cooler Water Outlet Temperature',
    'Stator core temperature', 'UGB metal temperature',
    'LGB metal temperature 1', 'LGB metal temperature 2',
    'LGB oil temperature', 'Penstock Flow', 'Turbine flow',
    'UGB cooling water flow', 'LGB cooling water flow',
    'Generator cooling water flow', 'Governor Penstock Pressure',
    'Penstock pressure', 'Opening Wicked Gate', 'UGB Oil Contaminant',
    'Gen Thrust Bearing Oil Contaminant']

df_data_withtime = df_noisy_wgn[['TimeStamp'] + feature_set]

In [16]:
df_data_withtime = df_noisy_wgn

In [17]:
df_data_withtime.shape

(1895590, 44)

In [18]:
df_label = pd.DataFrame({
    'TimeStamp': df_data_withtime['TimeStamp'],
    'label_anomaly': 0
})

for _, row in df_anomaly.iterrows():
    start_time = row['Start Time']
    end_time = row['End Time']
    pre_start_time = start_time - pd.Timedelta(hours=3)
    
    df_label.loc[
        (df_label['TimeStamp'] >= pre_start_time) & 
        (df_label['TimeStamp'] < start_time), 
        'label_anomaly'
    ] = 1
    
    # Remove timestamps between [start_time, end_time) from df_label and df_data_withtime
    mask_remove = (df_data_withtime['TimeStamp'] >= start_time) & (df_data_withtime['TimeStamp'] < end_time)
    df_data_withtime = df_data_withtime.loc[~mask_remove]
    df_label = df_label.loc[~mask_remove]

# Reset index of the resulting DataFrames if necessary
df_data_withtime.reset_index(drop=True, inplace=True)
df_label.reset_index(drop=True, inplace=True)


In [19]:
new_df = df_data_withtime

In [20]:
sampels = int(len(new_df) * 0.8)
train_data = new_df[:sampels]
test_data = new_df[sampels:]
test_label = df_label[sampels:]

train_data.reset_index(drop=True, inplace=True)
test_data.reset_index(drop=True, inplace=True)
test_label.reset_index(drop=True, inplace=True)

train_data.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/CLGS2AWGN30ES15/train.csv", index=False)
test_data.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/CLGS2AWGN30ES15/test.csv", index=False)
test_label.to_csv("/run/media/fourier/Data2/Pras/Vale/TN_Anom/DTAAD/data/CLGS2AWGN30ES15/test_label.csv", index=False)

In [21]:
import pandas as pd
import numpy as np
import os

In [22]:
def load_and_save(category, filename, dataset, dataset_folder):
    temp = np.genfromtxt(os.path.join(dataset_folder, category, filename),
                         dtype=np.float64,
                         delimiter=',')
    print(dataset, category, filename, temp.shape)
    np.save(os.path.join(output_folder, f"SMD/{dataset}_{category}.npy"), temp)
    return temp.shape

def load_and_save2(category, filename, dataset, dataset_folder, shape):
    temp = np.zeros(shape)
    with open(os.path.join(dataset_folder, 'interpretation_label', filename), "r") as f:
        ls = f.readlines()
    for line in ls:
        pos, values = line.split(':')[0], line.split(':')[1].split(',')
        start, end, indx = int(pos.split('-')[0]), int(pos.split('-')[1]), [int(i) - 1 for i in values]
        temp[start - 1:end - 1, indx] = 1
    print(dataset, category, filename, temp.shape)
    np.save(os.path.join(output_folder, f"SMD/{dataset}_{category}.npy"), temp)

def normalize3(a, min_a=None, max_a=None):
    if min_a is None: min_a, max_a = np.min(a, axis=0), np.max(a, axis=0)
    return ((a - min_a) / (max_a - min_a + 0.0001)), min_a, max_a


def convertNumpy(df):
    x = df[df.columns[3:]].values[::10, :]
    return (x - x.min(0)) / (x.ptp(0) + 1e-4)

In [23]:
dataset_folder = 'data/CLGS2AWGN30ES15'
df_train = pd.read_csv(os.path.join(dataset_folder, 'train.csv'))
df_test = pd.read_csv(os.path.join(dataset_folder, 'test.csv'))
df_train, df_test = df_train.values[:, 1:], df_test.values[:, 1:]
_, min_a, max_a = normalize3(np.concatenate((df_train, df_test), axis=0))
train, _, _ = normalize3(df_train, min_a, max_a)
test, _, _ = normalize3(df_test, min_a, max_a)
labels = pd.read_csv(os.path.join(dataset_folder, 'test_label.csv'))
labels = labels.values[:, 1:]

folder = os.path.join("processed", "CLGS2AWGN30ES15")
os.makedirs(folder, exist_ok=True)

for file in ['train', 'test', 'labels']:
    np.save(os.path.join(folder, f'{file}.npy'), eval(file).astype('float64'))